In [ ]:
import pandas as pd
bike_data=pd.read_csv('hour.csv')
print(bike_data.head(5))
print(bike_data.shape)
print(bike_data.columns)
print(bike_data.describe())
bike_data.info()
print(bike_data.dtypes)

In [ ]:
# Transforming DataColumns
bike_data['dteday']=pd.to_datetime(bike_data['dteday'])
print(bike_data['dteday'].dtype)
bike_data=bike_data.sort_values(by=['dteday','hr'])
bike_data.reset_index(drop=True)
print(bike_data[['dteday','hr','cnt']].head(10))

In [ ]:
import matplotlib.pyplot as plt
daily_rentals=bike_data.groupby('dteday')['cnt'].sum()
print(daily_rentals)
plt.figure(figsize=(15,5))
plt.plot(daily_rentals.index,daily_rentals.values)
plt.xlabel('Date')
plt.ylabel('Total Bike Rentals')
plt.title('Daily Bike Rentals Over Time')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
max_hours=bike_data.groupby('hr')['cnt'].mean()
busy_hr=max_hours.idxmax()
quiet_hr=max_hours.idxmin()
print(busy_hr)
print(quiet_hr)
avg_weekday = bike_data.groupby('weekday')['cnt'].mean()
avg_month = bike_data.groupby('mnth')['cnt'].mean()
avg_season = bike_data.groupby('season')['cnt'].mean()

print(avg_weekday)
print(avg_month)
print(avg_season)

In [ ]:
# Data Featuring
features = ['temp', 'hum', 'hr', 'weekday']
X = bike_data[features]
y = bike_data['cnt']
print(X.head())
print(y.head())
print(X.shape)
print(y.shape)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score
from sklearn.tree import DecisionTreeRegressor
import numpy as np
split=int(len(X)*0.8)
x_train=X.iloc[:split]
x_test=X.iloc[split:]
y_train=y.iloc[:split]
y_test=y.iloc[split:]
dtr=DecisionTreeRegressor(random_state=42)
dtr.fit(x_train,y_train)
dtr_pred=dtr.predict(x_test)
print("Actual values:", y_test.head(10).tolist())
print("Predicted values:", dtr_pred[:10])
mae = mean_absolute_error(y_test, dtr_pred)
mse = mean_squared_error(y_test, dtr_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, dtr_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

In [ ]:
# TimeSeriesSplit 
from sklearn.model_selection import TimeSeriesSplit
mae_scores = []
tscv=TimeSeriesSplit(n_splits=5)
for train_index,test_index in tscv.split(X):
    tsx_train,tsx_test=X.iloc[train_index],X.iloc[test_index]
    tsy_train,tsy_test=y.iloc[train_index],y.iloc[test_index]
    tsdt=DecisionTreeRegressor(random_state=42)
    tsdt.fit(tsx_train,tsy_train)
    predictions=tsdt.predict(tsx_test)
    mae = mean_absolute_error(tsy_test, predictions)
    mae_scores.append(mae)
print("MAE for each fold:", mae_scores)
print("Average MAE:", sum(mae_scores) / len(mae_scores))

In [50]:
# Rolling Window Split
from sklearn.ensemble import RandomForestRegressor
from dateutil.relativedelta import relativedelta
from sklearn.linear_model import LinearRegression 
start_date=bike_data['dteday'].min()
end_date=bike_data['dteday'].max()
current_date=start_date
ws_mae_scores=[]
lr_mae_scores = []
rf_mae_scores = []

while True:
    train_end=current_date+relativedelta(months=3)
    test_end=train_end+relativedelta(months=1)
    if test_end>end_date+pd.Timedelta(days=1):
        break
    train_data=bike_data[(bike_data['dteday']>=current_date) & (bike_data['dteday']<train_end)]
    test_data=bike_data[(bike_data['dteday']>=train_end) & (bike_data['dteday']<test_end)]
    print("Training:", current_date.date(),"to",(train_end-pd.Timedelta(days=1)).date())
    print("Testing:", train_end.date(),"to",(test_end-pd.Timedelta(days=1)).date())
    print("Train shape:", train_data.shape)
    print("Test shape:", test_data.shape)
    print("--------------------")
    ws_x_train=train_data[features]
    ws_y_train=train_data['cnt']
    ws_x_test=test_data[features]
    ws_y_test=test_data['cnt']
    ws_dct=DecisionTreeRegressor(random_state=42)
    ws_dct.fit(ws_x_train,ws_y_train)
    ws_predict= ws_dct.predict(ws_x_test)
    ws_mae=mean_absolute_error(ws_y_test,ws_predict)
    ws_mae_scores.append(ws_mae)
    print("MAE:", ws_mae)
    current_date+=relativedelta(months=1)
    lr_model = LinearRegression()
    lr_model.fit(ws_x_train, ws_y_train)
    
    lr_predictions = lr_model.predict(ws_x_test)
    
    lr_mae = mean_absolute_error(ws_y_test, lr_predictions)
    lr_mae_scores.append(lr_mae)
    rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
    )

    rf_model.fit(ws_x_train, ws_y_train)

    rf_predictions = rf_model.predict(ws_x_test)

    rf_mae = mean_absolute_error(ws_y_test, rf_predictions)
    rf_mae_scores.append(rf_mae)

print("Linear Regression MAE:", lr_mae)
print("MAE scores:", ws_mae_scores)
print("Average Rolling Window MAE:", sum(ws_mae_scores) / len(ws_mae_scores))
print("Average Linear Regression MAE:",
      sum(lr_mae_scores) / len(lr_mae_scores))
print("Average Random Forest MAE:",
      sum(rf_mae_scores) / len(rf_mae_scores))

Training: 2011-01-01 to 2011-03-31
Testing: 2011-04-01 to 2011-04-30
Train shape: (2067, 17)
Test shape: (719, 17)
--------------------
MAE: 50.217663421418635
Training: 2011-02-01 to 2011-04-30
Testing: 2011-05-01 to 2011-05-31
Train shape: (2098, 17)
Test shape: (744, 17)
--------------------
MAE: 52.928763440860216
Training: 2011-03-01 to 2011-05-31
Testing: 2011-06-01 to 2011-06-30
Train shape: (2193, 17)
Test shape: (720, 17)
--------------------
MAE: 42.50347222222222
Training: 2011-04-01 to 2011-06-30
Testing: 2011-07-01 to 2011-07-31
Train shape: (2183, 17)
Test shape: (744, 17)
--------------------
MAE: 42.19422043010753
Training: 2011-05-01 to 2011-07-31
Testing: 2011-08-01 to 2011-08-31
Train shape: (2208, 17)
Test shape: (731, 17)
--------------------
MAE: 41.127906976744185
Training: 2011-06-01 to 2011-08-31
Testing: 2011-09-01 to 2011-09-30
Train shape: (2195, 17)
Test shape: (717, 17)
--------------------
MAE: 47.818688981868895
Training: 2011-07-01 to 2011-09-30
Testing

In [ ]:
# Linear Regression
    lr_model = LinearRegression()
    lr_model.fit(ws_x_train, ws_y_train)
    
    lr_predictions = lr_model.predict(ws_x_test)
    
    lr_mae = mean_absolute_error(ws_y_test, lr_predictions)
    lr_mae_scores.append(lr_mae)
    
    print("Linear Regression MAE:", lr_mae)